In [0]:
%sql
-- UNDP · Paso 1 (estructura): cantidad de filas (≈ 206 países según S3).
SELECT COUNT(*) AS filas FROM happiness.bronze.undp;

In [0]:
%sql
-- UNDP · Paso 1 (estructura): agrupa las 1.115 columnas por indicador. Las columnas se llaman
-- <indicador>_<año> (ej. hdi_1990); se les saca el sufijo de año para contar cuántos años trae cada uno.
-- Las columnas sin año (iso3, country, hdicode, region...) quedan como su propio nombre.
SELECT REGEXP_REPLACE(column_name, '_[0-9]{4}$', '') AS indicador,
       COUNT(*)                                       AS columnas,
       MIN(ordinal_position)                          AS desde_posicion
FROM happiness.information_schema.columns
WHERE table_schema = 'bronze' AND table_name = 'undp'
GROUP BY ALL
ORDER BY desde_posicion;

In [0]:
%sql
-- UNDP · Paso 1 (estructura): muestra de las columnas de identificación y de unos pocos indicadores.
SELECT iso3, country, hdicode, region, hdi_rank_2023, hdi_1990, _ingest_ts, _source
FROM happiness.bronze.undp
LIMIT 5;

UNDP, paso 1: son 206 filas y 1.115 columnas, que se descomponen así:

8 columnas de identificación: iso3, country, hdicode, region, hdi_rank, gdi_group, gii_rank y rankdiff_hdi_phdi.
3 columnas de control: _rescued_data, _ingest_ts y _source.
36 indicadores con serie por años:
30 tienen 34 años, de 1990 a 2023 (por ejemplo hdi, le, eys, mys, gnipc, gii, mmr, co2_prod, pop_total).
6 tienen 14 años, probablemente de 2010 a 2023 (ihdi, coef_ineq, loss, ineq_le, ineq_edu, ineq_inc).
Muestra: Andorra viene con region vacía y hdi_1990 vacío.

Como la tabla es ancha, no tiene sentido mirar las 1.115 columnas. Primero vemos cuáles de los 36 indicadores usa el proyecto, y con eso armamos los pasos de rangos y años solo para esos. 

In [0]:
%sql
-- UNDP · Paso 2 (nulos): columnas de identificación y dos columnas de HDI (la más antigua y la más reciente).
-- Se separan NULL real, vacío y el texto 'null'. Andorra mostró region vacía en la muestra.
SELECT COUNT(*)                                          AS filas,
       COUNT_IF(iso3 IS NULL OR iso3 IN ('', 'null'))    AS iso3_faltante,
       COUNT_IF(country IS NULL OR country IN ('', 'null'))   AS pais_faltante,
       COUNT_IF(hdicode IS NULL)                         AS hdicode_null_real,
       COUNT_IF(hdicode IN ('', 'null'))                 AS hdicode_vacio_o_texto,
       COUNT_IF(region IS NULL)                          AS region_null_real,
       COUNT_IF(region IN ('', 'null'))                  AS region_vacia_o_texto,
       COUNT_IF(hdi_1990 IS NULL)                        AS hdi_1990_null_real,
       COUNT_IF(hdi_1990 IN ('', 'null'))                AS hdi_1990_vacio_o_texto,
       COUNT_IF(hdi_2023 IS NULL)                        AS hdi_2023_null_real,
       COUNT_IF(_rescued_data IS NOT NULL)               AS filas_rescatadas
FROM happiness.bronze.undp;

In [0]:
%sql
-- UNDP · Pasos 3 y 5 (cardinalidad y duplicados): clave natural = iso3.
-- Esperado: 206 iso3 distintos, 206 países distintos y 0 repetidos.
WITH numerado AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY iso3 ORDER BY _ingest_ts) AS rn
  FROM happiness.bronze.undp
)
SELECT COUNT(*)                  AS filas,
       COUNT(DISTINCT iso3)      AS iso3_distintos,
       COUNT(DISTINCT country)   AS paises_distintos,
       COUNT_IF(rn > 1)          AS claves_duplicadas
FROM numerado;

In [0]:
%sql
-- UNDP · Paso 3 (cardinalidad): categorías de desarrollo humano y regiones UNDP, con cuántos países hay en cada una.
-- Sirve para ver si hay categorías inesperadas (por ejemplo, filas de grupos o agregados mezclados con países).
SELECT hdicode, region, COUNT(*) AS paises
FROM happiness.bronze.undp
GROUP BY ALL
ORDER BY hdicode, region;

In [0]:
%sql
-- UNDP · Para dim_pais_base: filas de UNDP cuyo iso3 no existe en Banco Mundial.
-- Aquí se ven los agregados (si los hay) y los países con código distinto.
-- También permite revisar la codificación de los nombres (el archivo es ISO-8859-1): buscar tildes rotas.
SELECT u.iso3, u.country, u.hdicode, u.region
FROM happiness.bronze.undp u
LEFT JOIN (SELECT DISTINCT `Country Code` AS codigo FROM happiness.bronze.banco_mundial) bm
       ON u.iso3 = bm.codigo
WHERE bm.codigo IS NULL
ORDER BY u.iso3;

In [0]:
%sql
-- UNDP · Qué indicadores usa el proyecto: los 6 incluidos en config_indicadores para esta fuente.
-- Con esto sabemos qué columnas <indicador>_<año> importan, entre las 1.115 de la tabla.
SELECT fuente, codigo, nombre, tema, unidad, incluido
FROM happiness.meta.config_indicadores
-- WHERE fuente = 'UNDP'
-- ORDER BY incluido DESC, codigo;

In [0]:
%sql
-- UNDP · Cobertura y rango de los 5 indicadores confirmados + 3 candidatos a sexto (ihdi, gdi, gii).
-- paises_AAAA = cuántos de los 206 países tienen dato ese año (sin NULL, vacío ni 'null').
-- UNDP termina en 2023: con arrastre de 2 años (D3) alcanza justo para cubrir 2024 y 2025.
-- no_numericos_2023 debe ser 0 (si no, el CAST de Silver fallaría).
SELECT 'hdi' AS indicador, COUNT_IF(hdi_2011 IS NOT NULL AND hdi_2011 NOT IN ('', 'null')) AS paises_2011, COUNT_IF(hdi_2019 IS NOT NULL AND hdi_2019 NOT IN ('', 'null')) AS paises_2019, COUNT_IF(hdi_2023 IS NOT NULL AND hdi_2023 NOT IN ('', 'null')) AS paises_2023,
       MIN(TRY_CAST(hdi_2023 AS DOUBLE)) AS min_2023, MAX(TRY_CAST(hdi_2023 AS DOUBLE)) AS max_2023,
       COUNT_IF(hdi_2023 NOT IN ('', 'null') AND TRY_CAST(hdi_2023 AS DOUBLE) IS NULL) AS no_numericos_2023
FROM happiness.bronze.undp
UNION ALL
SELECT 'le' AS indicador, COUNT_IF(le_2011 IS NOT NULL AND le_2011 NOT IN ('', 'null')) AS paises_2011, COUNT_IF(le_2019 IS NOT NULL AND le_2019 NOT IN ('', 'null')) AS paises_2019, COUNT_IF(le_2023 IS NOT NULL AND le_2023 NOT IN ('', 'null')) AS paises_2023,
       MIN(TRY_CAST(le_2023 AS DOUBLE)) AS min_2023, MAX(TRY_CAST(le_2023 AS DOUBLE)) AS max_2023,
       COUNT_IF(le_2023 NOT IN ('', 'null') AND TRY_CAST(le_2023 AS DOUBLE) IS NULL) AS no_numericos_2023
FROM happiness.bronze.undp
UNION ALL
SELECT 'eys' AS indicador, COUNT_IF(eys_2011 IS NOT NULL AND eys_2011 NOT IN ('', 'null')) AS paises_2011, COUNT_IF(eys_2019 IS NOT NULL AND eys_2019 NOT IN ('', 'null')) AS paises_2019, COUNT_IF(eys_2023 IS NOT NULL AND eys_2023 NOT IN ('', 'null')) AS paises_2023,
       MIN(TRY_CAST(eys_2023 AS DOUBLE)) AS min_2023, MAX(TRY_CAST(eys_2023 AS DOUBLE)) AS max_2023,
       COUNT_IF(eys_2023 NOT IN ('', 'null') AND TRY_CAST(eys_2023 AS DOUBLE) IS NULL) AS no_numericos_2023
FROM happiness.bronze.undp
UNION ALL
SELECT 'mys' AS indicador, COUNT_IF(mys_2011 IS NOT NULL AND mys_2011 NOT IN ('', 'null')) AS paises_2011, COUNT_IF(mys_2019 IS NOT NULL AND mys_2019 NOT IN ('', 'null')) AS paises_2019, COUNT_IF(mys_2023 IS NOT NULL AND mys_2023 NOT IN ('', 'null')) AS paises_2023,
       MIN(TRY_CAST(mys_2023 AS DOUBLE)) AS min_2023, MAX(TRY_CAST(mys_2023 AS DOUBLE)) AS max_2023,
       COUNT_IF(mys_2023 NOT IN ('', 'null') AND TRY_CAST(mys_2023 AS DOUBLE) IS NULL) AS no_numericos_2023
FROM happiness.bronze.undp
UNION ALL
SELECT 'gnipc' AS indicador, COUNT_IF(gnipc_2011 IS NOT NULL AND gnipc_2011 NOT IN ('', 'null')) AS paises_2011, COUNT_IF(gnipc_2019 IS NOT NULL AND gnipc_2019 NOT IN ('', 'null')) AS paises_2019, COUNT_IF(gnipc_2023 IS NOT NULL AND gnipc_2023 NOT IN ('', 'null')) AS paises_2023,
       MIN(TRY_CAST(gnipc_2023 AS DOUBLE)) AS min_2023, MAX(TRY_CAST(gnipc_2023 AS DOUBLE)) AS max_2023,
       COUNT_IF(gnipc_2023 NOT IN ('', 'null') AND TRY_CAST(gnipc_2023 AS DOUBLE) IS NULL) AS no_numericos_2023
FROM happiness.bronze.undp
UNION ALL
SELECT 'ihdi' AS indicador, COUNT_IF(ihdi_2011 IS NOT NULL AND ihdi_2011 NOT IN ('', 'null')) AS paises_2011, COUNT_IF(ihdi_2019 IS NOT NULL AND ihdi_2019 NOT IN ('', 'null')) AS paises_2019, COUNT_IF(ihdi_2023 IS NOT NULL AND ihdi_2023 NOT IN ('', 'null')) AS paises_2023,
       MIN(TRY_CAST(ihdi_2023 AS DOUBLE)) AS min_2023, MAX(TRY_CAST(ihdi_2023 AS DOUBLE)) AS max_2023,
       COUNT_IF(ihdi_2023 NOT IN ('', 'null') AND TRY_CAST(ihdi_2023 AS DOUBLE) IS NULL) AS no_numericos_2023
FROM happiness.bronze.undp
UNION ALL
SELECT 'gdi' AS indicador, COUNT_IF(gdi_2011 IS NOT NULL AND gdi_2011 NOT IN ('', 'null')) AS paises_2011, COUNT_IF(gdi_2019 IS NOT NULL AND gdi_2019 NOT IN ('', 'null')) AS paises_2019, COUNT_IF(gdi_2023 IS NOT NULL AND gdi_2023 NOT IN ('', 'null')) AS paises_2023,
       MIN(TRY_CAST(gdi_2023 AS DOUBLE)) AS min_2023, MAX(TRY_CAST(gdi_2023 AS DOUBLE)) AS max_2023,
       COUNT_IF(gdi_2023 NOT IN ('', 'null') AND TRY_CAST(gdi_2023 AS DOUBLE) IS NULL) AS no_numericos_2023
FROM happiness.bronze.undp
UNION ALL
SELECT 'gii' AS indicador, COUNT_IF(gii_2011 IS NOT NULL AND gii_2011 NOT IN ('', 'null')) AS paises_2011, COUNT_IF(gii_2019 IS NOT NULL AND gii_2019 NOT IN ('', 'null')) AS paises_2019, COUNT_IF(gii_2023 IS NOT NULL AND gii_2023 NOT IN ('', 'null')) AS paises_2023,
       MIN(TRY_CAST(gii_2023 AS DOUBLE)) AS min_2023, MAX(TRY_CAST(gii_2023 AS DOUBLE)) AS max_2023,
       COUNT_IF(gii_2023 NOT IN ('', 'null') AND TRY_CAST(gii_2023 AS DOUBLE) IS NULL) AS no_numericos_2023
FROM happiness.bronze.undp;

In [0]:
%sql
-- UNDP · Las 2 filas con hdicode 'Other Countries or Territories' y las que no tienen HDI 2023.
-- Sirve para saber qué territorios son y si están en las otras fuentes (Banco Mundial, WHR).
SELECT iso3, country, hdicode, region, hdi_2023, le_2023, gnipc_2023
FROM happiness.bronze.undp
WHERE hdicode = 'Other Countries or Territories' OR hdi_2023 IS NULL
ORDER BY iso3;

## Conclusiones UNDP (bronze.undp, 206 filas)
- 195 países/territorios + 11 agregados (ZZA.VHHD ... ZZK.WORLD). Clave iso3 sin duplicados ni nulos.
- Los 195 iso3 existen en Banco Mundial (ISO3 estándar). Sin Taiwán.
- Región UNDP vacía en 43 países Very High, 1 territorio y los 11 agregados: usar la región de Banco Mundial.
- Rangos 2023 coherentes; 0 valores no numéricos. 36 indicadores: 30 con 1990-2023, 6 con 14 años.
- config_indicadores no tenía filas de UNDP: se registran los 6 (hdi, le, eys, mys, gnipc + sexto a decidir). Ver D25.